In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade torch torchvision transformers datasets timm pillow scikit-learn
!pip install -q peft accelerate bitsandbytes
!pip install --upgrade --force-reinstall huggingface_hub -q
!pip install optuna
!pip install git+https://github.com/openai/CLIP.git
!pip install ftfy regex tqdm

In [ ]:
import os
import pandas as pd

# Verify paths
base_path = '/kaggle/input/ml-challenge-2-2025/'
print(f"Train CSV: {os.path.exists(base_path + 'train.csv')}")
print(f"Test CSV: {os.path.exists(base_path + 'test.csv')}")
print(f"Images: {os.path.exists(base_path + 'images/')}")

# Check data
train_df = pd.read_csv(base_path + 'train.csv')
test_df = pd.read_csv(base_path + 'test.csv')
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")

# Count images
images = [f for f in os.listdir(base_path + 'images/') if f.endswith(('.jpg', '.png'))]
print(f"\nTotal images: {len(images)}")
print(f"Sample images: {images[:3]}")

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Add your HF token in Kaggle Secrets first (Add-ons -> Secrets -> Add: HF_TOKEN)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
# Check if transformers installed correctly
import transformers
import torch
import accelerate
from PIL import Image

print(f"✅ transformers: {transformers.__version__}")
print(f"✅ torch: {torch.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")
print(f"✅ PIL (pillow): {Image.__version__ if hasattr(Image, '__version__') else 'Installed'}")
print("\n🎉 All packages installed successfully!")

In [ ]:
"""
STEP 1: ADVANCED TEXT EMBEDDINGS EXTRACTION
===========================================
Creates semantic embeddings using Sentence Transformers
+ Statistical text features for comprehensive representation
Target: Better semantic understanding than TF-IDF alone
"""

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import torch
import gc
import os

# ==================== CONFIGURATION ====================
CONFIG = {
    'train_csv': '/kaggle/input/ml-challenge-2-2025/train.csv',
    'test_csv': '/kaggle/input/ml-challenge-2-2025/test.csv',
    
    # Output paths
    'train_title_emb': '/kaggle/working/train_title_embeddings.npy',
    'test_title_emb': '/kaggle/working/test_title_embeddings.npy',
    'train_full_emb': '/kaggle/working/train_full_embeddings.npy',
    'test_full_emb': '/kaggle/working/test_full_embeddings.npy',
    'train_stats': '/kaggle/working/train_text_stats.npy',
    'test_stats': '/kaggle/working/test_text_stats.npy',
    
    # Model settings
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',  # Fast, 384-dim
    # Alternative: 'sentence-transformers/all-mpnet-base-v2' (768-dim, slower but better)
    'batch_size': 64,
    'max_length': 256,
}

class TextEmbeddingExtractor:
    """Extract comprehensive text embeddings"""
    
    def __init__(self):
        print("🔧 Initializing embedding model...")
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"   Device: {self.device}")
        
        # Load sentence transformer
        self.model = SentenceTransformer(CONFIG['embedding_model'])
        self.model.to(self.device)
        self.model.eval()
        
        print(f"✓ Model loaded: {CONFIG['embedding_model']}")
        print(f"✓ Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
    
    def extract_title(self, text):
        """Extract title from catalog content"""
        if pd.isna(text):
            return ""
        
        text_str = str(text)
        
        # Try to extract Item Name
        import re
        match = re.search(r'Item Name:\s*([^\n]+)', text_str)
        if match:
            return match.group(1).strip()
        
        # Fallback: first line or first 100 chars
        first_line = text_str.split('\n')[0]
        return first_line[:100] if len(first_line) > 100 else first_line
    
    def truncate_text(self, text, max_words=150):
        """Truncate text to manageable length"""
        if pd.isna(text):
            return ""
        
        words = str(text).split()
        return ' '.join(words[:max_words])
    
    def compute_embeddings_batch(self, texts, desc="Encoding"):
        """Compute embeddings in batches"""
        embeddings = []
        
        for i in tqdm(range(0, len(texts), CONFIG['batch_size']), desc=desc):
            batch = texts[i:i + CONFIG['batch_size']]
            
            with torch.no_grad():
                batch_embeddings = self.model.encode(
                    batch,
                    batch_size=CONFIG['batch_size'],
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=True  # L2 normalize
                )
            
            embeddings.append(batch_embeddings)
            
            # Memory management
            if i % (CONFIG['batch_size'] * 10) == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        
        return np.vstack(embeddings)
    
    def compute_statistical_features(self, texts):
        """Compute statistical text features"""
        print("   Computing statistical features...")
        
        features = []
        
        for text in tqdm(texts, desc="Text stats"):
            if pd.isna(text):
                text = ""
            else:
                text = str(text)
            
            words = text.split()
            chars = len(text)
            
            feature_vec = [
                # Basic stats
                len(text),  # char count
                len(words),  # word count
                len(set(words)) / max(len(words), 1),  # unique word ratio
                np.mean([len(w) for w in words]) if words else 0,  # avg word length
                
                # Character ratios
                sum(c.isdigit() for c in text) / max(chars, 1),  # digit ratio
                sum(c.isupper() for c in text) / max(chars, 1),  # uppercase ratio
                sum(c.isalpha() for c in text) / max(chars, 1),  # alpha ratio
                text.count(' ') / max(chars, 1),  # space ratio
                
                # Punctuation
                text.count(',') / max(chars, 1),
                text.count('.') / max(chars, 1),
                text.count('-') / max(chars, 1),
                text.count('(') / max(chars, 1),
                
                # Structure indicators
                text.count('\n'),  # line breaks
                int('bullet point' in text.lower()),
                int('item name' in text.lower()),
                int('value:' in text.lower()),
                
                # Number features
                len([w for w in words if w.isdigit()]),  # numeric word count
                
                # Length bins (one-hot encoded)
                int(chars < 200),
                int(200 <= chars < 500),
                int(500 <= chars < 1000),
                int(chars >= 1000),
            ]
            
            features.append(feature_vec)
        
        return np.array(features)
    
    def process_dataset(self, df, dataset_name='Dataset'):
        """Process entire dataset"""
        print(f"\n{'='*60}")
        print(f"📝 Processing {dataset_name}")
        print(f"{'='*60}")
        
        texts = df['catalog_content'].fillna('')
        
        # Extract titles
        print("   Extracting titles...")
        titles = [self.extract_title(t) for t in tqdm(texts, desc="Titles")]
        
        # Truncate full text
        print("   Truncating full texts...")
        full_texts = [self.truncate_text(t) for t in tqdm(texts, desc="Truncate")]
        
        # Compute embeddings
        print("\n   Computing title embeddings...")
        title_embeddings = self.compute_embeddings_batch(titles, "Title embeddings")
        
        print("   Computing full text embeddings...")
        full_embeddings = self.compute_embeddings_batch(full_texts, "Full text embeddings")
        
        # Compute statistical features
        statistical_features = self.compute_statistical_features(texts)
        
        print(f"\n✓ Title embeddings: {title_embeddings.shape}")
        print(f"✓ Full embeddings: {full_embeddings.shape}")
        print(f"✓ Statistical features: {statistical_features.shape}")
        
        return title_embeddings, full_embeddings, statistical_features

def main():
    """Main extraction pipeline"""
    print("="*80)
    print("🚀 ADVANCED TEXT EMBEDDINGS EXTRACTION")
    print("="*80)
    print(f"Model: {CONFIG['embedding_model']}")
    print(f"Batch size: {CONFIG['batch_size']}")
    print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
    print("="*80)
    
    # Load datasets
    print("\n[1/3] Loading datasets...")
    train_df = pd.read_csv(CONFIG['train_csv'])
    test_df = pd.read_csv(CONFIG['test_csv'])
    print(f"✓ Train: {len(train_df):,} samples")
    print(f"✓ Test: {len(test_df):,} samples")
    
    # Initialize extractor
    print("\n[2/3] Initializing extractor...")
    extractor = TextEmbeddingExtractor()
    
    # Process train set
    print("\n[3/3] Extracting embeddings...")
    train_title_emb, train_full_emb, train_stats = extractor.process_dataset(
        train_df, 'Training Set'
    )
    
    # Process test set
    test_title_emb, test_full_emb, test_stats = extractor.process_dataset(
        test_df, 'Test Set'
    )
    
    # Save embeddings
    print("\n💾 Saving embeddings...")
    np.save(CONFIG['train_title_emb'], train_title_emb)
    np.save(CONFIG['test_title_emb'], test_title_emb)
    np.save(CONFIG['train_full_emb'], train_full_emb)
    np.save(CONFIG['test_full_emb'], test_full_emb)
    np.save(CONFIG['train_stats'], train_stats)
    np.save(CONFIG['test_stats'], test_stats)
    
    print(f"✓ Saved: {CONFIG['train_title_emb']}")
    print(f"✓ Saved: {CONFIG['test_title_emb']}")
    print(f"✓ Saved: {CONFIG['train_full_emb']}")
    print(f"✓ Saved: {CONFIG['test_full_emb']}")
    print(f"✓ Saved: {CONFIG['train_stats']}")
    print(f"✓ Saved: {CONFIG['test_stats']}")
    
    # Summary
    total_features = (
        train_title_emb.shape[1] + 
        train_full_emb.shape[1] + 
        train_stats.shape[1]
    )
    
    print("\n" + "="*80)
    print("🎉 EMBEDDING EXTRACTION COMPLETE!")
    print("="*80)
    print(f"📊 Feature Breakdown:")
    print(f"   Title embeddings: {train_title_emb.shape[1]}D")
    print(f"   Full text embeddings: {train_full_emb.shape[1]}D")
    print(f"   Statistical features: {train_stats.shape[1]}D")
    print(f"   TOTAL: {total_features}D")
    print("\n💡 Next step: Run Step 2 to train models with these embeddings")
    print("="*80)
    
    return {
        'train_title': train_title_emb,
        'test_title': test_title_emb,
        'train_full': train_full_emb,
        'test_full': test_full_emb,
        'train_stats': train_stats,
        'test_stats': test_stats,
    }

if __name__ == "__main__":
    embeddings = main()

In [ ]:
"""
STEP 2: MODEL TRAINING WITH SEMANTIC EMBEDDINGS
================================================
Combines:
- Sentence transformer embeddings (semantic understanding)
- Your proven engineered features (46.2% baseline)
- Optimized stacking ensemble
Target: Beat 46.2% SMAPE significantly
"""

import pandas as pd
import numpy as np
import re
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.decomposition import TruncatedSVD
import warnings
import os
warnings.filterwarnings('ignore')

# ==================== CONFIGURATION ====================
CONFIG = {
    'train_csv': '/kaggle/input/ml-challenge-2-2025/train.csv',
    'test_csv': '/kaggle/input/ml-challenge-2-2025/test.csv',
    
    # Embedding paths (from Step 1)
    'train_title_emb': '/kaggle/working/train_title_embeddings.npy',
    'test_title_emb': '/kaggle/working/test_title_embeddings.npy',
    'train_full_emb': '/kaggle/working/train_full_embeddings.npy',
    'test_full_emb': '/kaggle/working/test_full_embeddings.npy',
    'train_stats': '/kaggle/working/train_text_stats.npy',
    'test_stats': '/kaggle/working/test_text_stats.npy',
    
    'output_path': '/kaggle/working/test_out.csv',
    
    # Model settings
    'n_folds': 7,  # More folds for stability
    'random_state': 42,
    
    # Feature settings
    'use_tfidf': True,  # Still use TF-IDF for keyword signals
    'tfidf_max_features': 500,  # Reduced since we have embeddings
    'use_svd_on_embeddings': True,  # Dimensionality reduction
    'svd_components': 50,  # Reduce embedding dimensions
}

# ==================== SMAPE METRIC ====================
def calculate_smape(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    predicted = np.maximum(predicted, 0.01)
    numerator = np.abs(predicted - actual)
    denominator = (np.abs(actual) + np.abs(predicted)) / 2
    smape = np.mean(numerator / (denominator + 1e-10)) * 100
    return smape

# ==================== YOUR PROVEN FEATURE EXTRACTION ====================
def extract_value_field(text):
    if pd.isna(text):
        return np.nan
    match = re.search(r'Value:\s*(\d+\.?\d*)', str(text))
    return float(match.group(1)) if match else np.nan

def extract_pack_info_comprehensive(text):
    if pd.isna(text):
        return 1, 'none'
    text_str = str(text).lower()
    value_match = re.search(r'value:\s*(\d+\.?\d*)', text_str)
    if value_match:
        val = float(value_match.group(1))
        if val > 1:
            return val, 'value_field'
    pack_patterns = [
        r'\(pack of (\d+)\)', r'pack of (\d+)', r'\((\d+) pack\)', 
        r'(\d+) pack', r'(\d+)-pack'
    ]
    for pattern in pack_patterns:
        match = re.search(pattern, text_str)
        if match:
            return float(match.group(1)), 'pack'
    case_match = re.search(r'(\d+)\s*per case|case of (\d+)', text_str)
    if case_match:
        for group in case_match.groups():
            if group:
                return float(group), 'case'
    count_match = re.search(r'(\d+)\s*count', text_str)
    if count_match:
        count = float(count_match.group(1))
        if count > 1:
            return count, 'count'
    return 1, 'single'

def extract_unit_info(text):
    if pd.isna(text):
        return 0, 'none', 0
    text_str = str(text).lower()
    patterns = [
        (r'(\d+\.?\d*)\s*ounce', 'ounce', 1),
        (r'(\d+\.?\d*)\s*oz', 'ounce', 1),
        (r'(\d+\.?\d*)\s*fl oz', 'fl_oz', 1),
        (r'(\d+\.?\d*)\s*pound', 'pound', 16),
        (r'(\d+\.?\d*)\s*lb', 'pound', 16),
        (r'(\d+\.?\d*)\s*ml', 'ml', 0.033814),
        (r'(\d+\.?\d*)\s*liter', 'liter', 33.814),
        (r'(\d+\.?\d*)\s*gram', 'gram', 0.035274),
        (r'(\d+\.?\d*)\s*kg', 'kg', 35.274),
    ]
    for pattern, unit, conversion in patterns:
        match = re.search(pattern, text_str)
        if match:
            value = float(match.group(1))
            return value, unit, value * conversion
    return 0, 'none', 0

def create_advanced_features(df):
    """Your proven feature engineering"""
    features = pd.DataFrame()
    text = df['catalog_content'].fillna('')
    text_lower = text.str.lower()
    
    # Pack and unit features
    pack_info = text.apply(extract_pack_info_comprehensive)
    features['pack_quantity'] = pack_info.apply(lambda x: x[0])
    pack_type_map = {'none': 0, 'single': 1, 'value_field': 2, 'pack': 3, 'case': 4, 'count': 5}
    features['pack_type_encoded'] = pack_info.apply(lambda x: x[1]).map(pack_type_map).fillna(0)
    features['is_multi_pack'] = (features['pack_quantity'] > 1).astype(int)
    features['log_pack_qty'] = np.log1p(features['pack_quantity'])
    
    unit_info = text.apply(extract_unit_info)
    features['unit_size'] = unit_info.apply(lambda x: x[0])
    features['unit_oz_equiv'] = unit_info.apply(lambda x: x[2])
    features['has_unit_size'] = (features['unit_size'] > 0).astype(int)
    features['total_volume'] = features['pack_quantity'] * features['unit_oz_equiv']
    features['log_total_volume'] = np.log1p(features['total_volume'])
    features['volume_per_pack'] = features['total_volume'] / (features['pack_quantity'] + 1)
    
    # Value field
    features['value_field'] = text.apply(extract_value_field)
    features['has_value_field'] = (~features['value_field'].isna()).astype(int)
    features['value_field_filled'] = features['value_field'].fillna(1)
    features['value_to_pack_ratio'] = features['value_field_filled'] / (features['pack_quantity'] + 1)
    
    # Brand indicators
    premium_brands = ['organic', 'premium', 'gourmet', 'artisan', 'handcrafted', 'imported', 'specialty', 'luxury']
    budget_brands = ['value', 'basic', 'economy', 'budget', 'generic']
    features['is_premium'] = text_lower.apply(lambda x: any(b in x for b in premium_brands)).astype(int)
    features['is_budget'] = text_lower.apply(lambda x: any(b in x for b in budget_brands)).astype(int)
    
    # Text structure
    features['text_length'] = text.str.len()
    features['word_count'] = text.str.split().str.len()
    features['bullet_count'] = text.str.count('Bullet Point')
    features['has_bullets'] = (features['bullet_count'] > 0).astype(int)
    
    titles = text.str.extract(r'Item Name:([^\n]+)', expand=False).fillna('')
    features['title_length'] = titles.str.len()
    features['title_to_total_ratio'] = features['title_length'] / (features['text_length'] + 1)
    
    # Number features
    all_numbers = text.apply(lambda x: [float(n) for n in re.findall(r'\d+\.?\d*', str(x))] if pd.notna(x) else [])
    features['num_count'] = all_numbers.apply(len)
    features['num_density'] = features['num_count'] / (features['text_length'] + 1) * 1000
    features['max_number'] = all_numbers.apply(lambda x: max(x) if x else 0)
    features['sum_numbers'] = all_numbers.apply(lambda x: sum(x) if x else 0)
    features['has_small_nums'] = all_numbers.apply(lambda x: any(0.1 <= n <= 10 for n in x)).astype(int)
    features['has_medium_nums'] = all_numbers.apply(lambda x: any(10 < n <= 100 for n in x)).astype(int)
    features['has_large_nums'] = all_numbers.apply(lambda x: any(n > 100 for n in x)).astype(int)
    
    # Keyword features
    keywords = {
        'bulk': ['bulk', 'wholesale', 'case', 'carton'],
        'individual': ['single', 'individual', 'piece', 'unit'],
        'family': ['family size', 'family pack', 'party size'],
        'trial': ['sample', 'trial', 'mini', 'travel'],
    }
    for category, words in keywords.items():
        features[f'is_{category}'] = text_lower.apply(lambda x: any(w in x for w in words)).astype(int)
    
    # Category hints
    categories = {
        'food': ['food', 'edible', 'snack', 'beverage', 'drink'],
        'beauty': ['beauty', 'cosmetic', 'skincare', 'makeup'],
        'household': ['cleaning', 'detergent', 'paper', 'towel'],
    }
    for cat, words in categories.items():
        features[f'cat_{cat}'] = text_lower.apply(lambda x: any(w in x for w in words)).astype(int)
    
    # Interactions
    features['volume_x_premium'] = features['log_total_volume'] * features['is_premium']
    features['pack_x_bulk'] = features['log_pack_qty'] * features['is_bulk']
    
    return features.fillna(0)

def add_target_encoded_features(train_df, test_df, target_col='price'):
    """Target encoding"""
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    
    train_df['first_word'] = train_df['catalog_content'].str.split().str[0].fillna('UNKNOWN')
    test_df['first_word'] = test_df['catalog_content'].str.split().str[0].fillna('UNKNOWN')
    
    train_df['potential_brand'] = train_df['catalog_content'].str.extract(
        r'Item Name:\s*([A-Z][a-z]+)', expand=False
    ).fillna('UNKNOWN')
    test_df['potential_brand'] = test_df['catalog_content'].str.extract(
        r'Item Name:\s*([A-Z][a-z]+)', expand=False
    ).fillna('UNKNOWN')
    
    categorical_features = ['first_word', 'potential_brand']
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    for cat_col in categorical_features:
        train_df[f'{cat_col}_target_enc'] = 0.0
        global_mean = train_df[target_col].mean()
        
        for train_idx, val_idx in kf.split(train_df):
            means = train_df.iloc[train_idx].groupby(cat_col)[target_col].mean()
            train_df.iloc[val_idx, train_df.columns.get_loc(f'{cat_col}_target_enc')] = \
                train_df.iloc[val_idx][cat_col].map(means).fillna(global_mean).values
        
        category_means = train_df.groupby(cat_col)[target_col].mean()
        test_df[f'{cat_col}_target_enc'] = test_df[cat_col].map(category_means).fillna(global_mean)
        
        category_counts = train_df.groupby(cat_col).size()
        smooth_means = (category_means * category_counts + global_mean) / (category_counts + 1)
        train_df[f'{cat_col}_smooth_enc'] = train_df[cat_col].map(smooth_means).fillna(global_mean)
        test_df[f'{cat_col}_smooth_enc'] = test_df[cat_col].map(smooth_means).fillna(global_mean)
    
    train_df = train_df.drop(categorical_features, axis=1)
    test_df = test_df.drop(categorical_features, axis=1)
    
    return train_df, test_df

def create_light_tfidf(train_texts, test_texts):
    """Lightweight TF-IDF for keyword signals"""
    tfidf = TfidfVectorizer(
        max_features=CONFIG['tfidf_max_features'],
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.95,
        sublinear_tf=True,
    )
    
    train_tfidf = tfidf.fit_transform(train_texts).toarray()
    test_tfidf = tfidf.transform(test_texts).toarray()
    
    return train_tfidf, test_tfidf

# ==================== EMBEDDING LOADING ====================
def load_embeddings():
    """Load precomputed embeddings"""
    print("\n📥 Loading embeddings...")
    
    embeddings = {}
    
    # Check if files exist
    required_files = [
        ('train_title_emb', CONFIG['train_title_emb']),
        ('test_title_emb', CONFIG['test_title_emb']),
        ('train_full_emb', CONFIG['train_full_emb']),
        ('test_full_emb', CONFIG['test_full_emb']),
        ('train_stats', CONFIG['train_stats']),
        ('test_stats', CONFIG['test_stats']),
    ]
    
    for name, path in required_files:
        if os.path.exists(path):
            embeddings[name] = np.load(path)
            print(f"✓ {name}: {embeddings[name].shape}")
        else:
            print(f"❌ {name} not found at {path}")
            print("   Run Step 1 first to generate embeddings!")
            return None
    
    # Optional: Apply SVD for dimensionality reduction
    if CONFIG['use_svd_on_embeddings']:
        print("\n🔄 Applying SVD dimensionality reduction...")
        
        for emb_type in ['title_emb', 'full_emb']:
            train_key = f'train_{emb_type}'
            test_key = f'test_{emb_type}'
            
            if embeddings[train_key].shape[1] > CONFIG['svd_components']:
                svd = TruncatedSVD(n_components=CONFIG['svd_components'], random_state=42)
                embeddings[train_key] = svd.fit_transform(embeddings[train_key])
                embeddings[test_key] = svd.transform(embeddings[test_key])
                print(f"✓ {emb_type}: {embeddings[train_key].shape[1]}D (SVD reduced)")
    
    return embeddings

# ==================== MODEL PARAMETERS ====================
lgbm_params = {
    'objective': 'regression',
    'metric': 'mae',
    'n_estimators': 5000,
    'learning_rate': 0.008,
    'max_depth': 10,
    'num_leaves': 80,
    'min_child_samples': 20,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

xgb_params = {
    'objective': 'reg:squarederror',
    'n_estimators': 5000,
    'learning_rate': 0.008,
    'max_depth': 9,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'gamma': 0.1,
    'random_state': 43,
    'tree_method': 'hist',
    'n_jobs': -1,
    'early_stopping_rounds': 300
}

catboost_params = {
    'iterations': 5000,
    'learning_rate': 0.008,
    'depth': 8,
    'l2_leaf_reg': 3.0,
    'random_seed': 44,
    'loss_function': 'MAE',
    'verbose': False
}

# ==================== STACKING ENSEMBLE ====================
def train_stacking_ensemble(X_train, y_train, X_test, n_folds=7):
    """Train stacking ensemble"""
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    oof_lgbm = np.zeros(len(X_train))
    oof_xgb = np.zeros(len(X_train))
    oof_cat = np.zeros(len(X_train))
    
    test_lgbm = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))
    test_cat = np.zeros(len(X_test))
    
    fold_smapes = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
        print(f"\n{'='*70}\n📊 FOLD {fold}/{n_folds}\n{'='*70}")
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # LightGBM
        print("Training LightGBM...")
        lgbm = lgb.LGBMRegressor(**lgbm_params)
        lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                 callbacks=[lgb.early_stopping(300, verbose=False)])
        oof_lgbm[val_idx] = lgbm.predict(X_val)
        test_lgbm += lgbm.predict(X_test) / n_folds
        lgbm_smape = calculate_smape(np.expm1(y_val), np.expm1(oof_lgbm[val_idx]))
        print(f"  SMAPE: {lgbm_smape:.2f}%")
        
        # XGBoost
        print("Training XGBoost...")
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = xgb_model.predict(X_val)
        test_xgb += xgb_model.predict(X_test) / n_folds
        xgb_smape = calculate_smape(np.expm1(y_val), np.expm1(oof_xgb[val_idx]))
        print(f"  SMAPE: {xgb_smape:.2f}%")
        
        # CatBoost
        print("Training CatBoost...")
        cat_model = CatBoostRegressor(**catboost_params)
        cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), 
                      early_stopping_rounds=300, verbose=False)
        oof_cat[val_idx] = cat_model.predict(X_val)
        test_cat += cat_model.predict(X_test) / n_folds
        cat_smape = calculate_smape(np.expm1(y_val), np.expm1(oof_cat[val_idx]))
        print(f"  SMAPE: {cat_smape:.2f}%")
        
        # Fold ensemble
        fold_pred = (oof_lgbm[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3
        fold_smape = calculate_smape(np.expm1(y_val), np.expm1(fold_pred))
        fold_smapes.append(fold_smape)
        print(f"\n🎯 Fold {fold} Average SMAPE: {fold_smape:.2f}%")
    
    # Meta-model
    print("\n" + "="*70)
    print("Training Meta-Model...")
    print("="*70)
    meta_features = np.column_stack([oof_lgbm, oof_xgb, oof_cat])
    meta_model = Ridge(alpha=1.0, random_state=42)
    meta_model.fit(meta_features, y_train)
    
    test_meta_features = np.column_stack([test_lgbm, test_xgb, test_cat])
    test_preds_log = meta_model.predict(test_meta_features)
    
    oof_preds_log = meta_model.predict(meta_features)
    oof_smape = calculate_smape(np.expm1(y_train), np.expm1(oof_preds_log))
    
    print(f"\n📊 Cross-Validation Results:")
    print(f"   Fold SMAPEs: {[f'{s:.2f}%' for s in fold_smapes]}")
    print(f"   Mean: {np.mean(fold_smapes):.2f}% ± {np.std(fold_smapes):.2f}%")
    print(f"\n🎯 Meta-Model OOF SMAPE: {oof_smape:.2f}%")
    print(f"\n📊 Meta-Model Weights:")
    print(f"   LGBM:     {meta_model.coef_[0]:.4f}")
    print(f"   XGB:      {meta_model.coef_[1]:.4f}")
    print(f"   CatBoost: {meta_model.coef_[2]:.4f}")
    
    return np.expm1(test_preds_log), oof_smape

# ==================== MAIN PIPELINE ====================
def main():
    print("="*80)
    print("🚀 SEMANTIC EMBEDDINGS + PROVEN FEATURES PIPELINE")
    print("="*80)
    print("Combining: Sentence Transformers + Engineered Features")
    print("="*80)
    
    # Load data
    print("\n[1/7] Loading data...")
    train_df = pd.read_csv(CONFIG['train_csv'])
    test_df = pd.read_csv(CONFIG['test_csv'])
    
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    print(f"✓ Train: {len(train_df):,} samples")
    print(f"✓ Test: {len(test_df):,} samples")
    
    train_df['catalog_content'] = train_df['catalog_content'].fillna('')
    test_df['catalog_content'] = test_df['catalog_content'].fillna('')
    
    # Load embeddings
    print("\n[2/7] Loading embeddings...")
    embeddings = load_embeddings()
    if embeddings is None:
        print("\n❌ Cannot proceed without embeddings. Run Step 1 first!")
        return
    
    # Create engineered features
    print("\n[3/7] Creating engineered features...")
    train_text_feat = create_advanced_features(train_df)
    test_text_feat = create_advanced_features(test_df)
    print(f"✓ Engineered features: {train_text_feat.shape[1]}")
    
    # Target encoding
    print("\n[4/7] Adding target-encoded features...")
    train_df_enc, test_df_enc = add_target_encoded_features(
        train_df.copy(), test_df.copy(), target_col='price'
    )
    target_enc_cols = [col for col in train_df_enc.columns if '_enc' in col]
    train_target_enc = train_df_enc[target_enc_cols].values
    test_target_enc = test_df_enc[target_enc_cols].values
    print(f"✓ Target-encoded features: {len(target_enc_cols)}")
    
    # TF-IDF (optional but helpful for keywords)
    if CONFIG['use_tfidf']:
        print("\n[5/7] Creating lightweight TF-IDF...")
        train_tfidf, test_tfidf = create_light_tfidf(
            train_df['catalog_content'], test_df['catalog_content']
        )
        print(f"✓ TF-IDF features: {train_tfidf.shape[1]}")
    else:
        print("\n[5/7] Skipping TF-IDF...")
        train_tfidf = np.array([]).reshape(len(train_df), 0)
        test_tfidf = np.array([]).reshape(len(test_df), 0)
    
    # Combine ALL features
    print("\n[6/7] Combining all features...")
    
    feature_components = [
        ('TF-IDF', train_tfidf, test_tfidf),
        ('Engineered', train_text_feat.values, test_text_feat.values),
        ('Target Encoding', train_target_enc, test_target_enc),
        ('Title Embeddings', embeddings['train_title_emb'], embeddings['test_title_emb']),
        ('Full Text Embeddings', embeddings['train_full_emb'], embeddings['test_full_emb']),
        ('Statistical Features', embeddings['train_stats'], embeddings['test_stats']),
    ]
    
    train_components = []
    test_components = []
    feature_breakdown = []
    
    for name, train_feat, test_feat in feature_components:
        if train_feat.shape[1] > 0:
            train_components.append(train_feat)
            test_components.append(test_feat)
            feature_breakdown.append(f"   - {name}: {train_feat.shape[1]}")
    
    X_train = np.hstack(train_components)
    X_test = np.hstack(test_components)
    
    print(f"✓ Final feature count: {X_train.shape[1]}")
    print("\n📊 Feature Breakdown:")
    for line in feature_breakdown:
        print(line)
    
    y_train = np.log1p(train_df['price'].values)
    
    # Train ensemble
    print("\n[7/7] Training stacking ensemble...")
    print("="*80)
    predictions, oof_smape = train_stacking_ensemble(
        X_train, y_train, X_test, CONFIG['n_folds']
    )
    
    # Save submission
    predictions = np.maximum(predictions, 0.1)
    submission = pd.DataFrame({
        'sample_id': test_df['sample_id'], 
        'price': predictions
    })
    submission.to_csv(CONFIG['output_path'], index=False)
    
    # Final results
    print("\n" + "="*80)
    print("✅ TRAINING COMPLETE!")
    print("="*80)
    print(f"\n🎯 Expected Leaderboard Score: ~{oof_smape:.1f}%")
    print(f"💾 Submission saved: {CONFIG['output_path']}")
    
    print(f"\n📈 Prediction Statistics:")
    print(f"   Count:  {len(predictions):,}")
    print(f"   Min:    ${predictions.min():.2f}")
    print(f"   Max:    ${predictions.max():.2f}")
    print(f"   Mean:   ${predictions.mean():.2f}")
    print(f"   Median: ${np.median(predictions):.2f}")
    
    print("\n🏆 COMPARISON:")
    print(f"   Baseline (TF-IDF only):    46.20% SMAPE")
    print(f"   This model (+ embeddings): {oof_smape:.2f}% SMAPE")
    
    if oof_smape < 46.2:
        improvement = 46.2 - oof_smape
        print(f"   🎉 IMPROVEMENT: {improvement:.2f}% better!")
        if oof_smape < 45:
            print(f"   🏆🏆🏆 SUB-45% ACHIEVED!")
        elif oof_smape < 46:
            print(f"   🏆 EXCELLENT: Sub-46% achieved!")
    else:
        print(f"   ⚠️  Embeddings didn't improve over baseline")
        print(f"   💡 This can happen - TF-IDF is sometimes stronger for e-commerce")
    
    print(f"\n📊 Model Details:")
    print(f"   Total features: {X_train.shape[1]}")
    print(f"   CV folds: {CONFIG['n_folds']}")
    print(f"   Models: LightGBM + XGBoost + CatBoost + Ridge Meta")
    
    print(f"\n💡 Key Advantages of This Approach:")
    print(f"   ✓ Semantic understanding from Sentence Transformers")
    print(f"   ✓ Proven engineered features (46.2% baseline)")
    print(f"   ✓ TF-IDF for keyword/token-level signals")
    print(f"   ✓ Target encoding for categorical information")
    print(f"   ✓ {CONFIG['n_folds']}-fold CV for robustness")
    print(f"   ✓ Stacking ensemble for optimal blending")
    
    if oof_smape >= 46:
        print(f"\n💡 Further Improvement Ideas:")
        print(f"   • Try different embedding models (all-mpnet-base-v2)")
        print(f"   • Add image embeddings (CLIP)")
        print(f"   • Create interaction features between embeddings and prices")
        print(f"   • Experiment with embedding dimensionality")
        print(f"   • Try weighted ensemble based on price ranges")
    
    print("="*80)
    
    return submission

if __name__ == "__main__":
    submission = main()